# 选股流程实战

本案例展示如何使用量化系统筛选符合条件的股票。

## 核心功能
- 获取主力资金流入TOP N
- 筛选DDX刚转正的股票
- 分级推荐（A/B/C/D级）
- 推送选股结果到钉钉

## 1. 环境准备

In [ ]:
import os
import sys
from pathlib import Path

# 添加项目路径
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / 'scripts'))

# 检查环境变量
print("=== 环境变量检查 ===")
print(f"妙想API Key: {'已配置' if os.environ.get('MX_APIKEY') else '未配置'}")
print(f"问财API Key: {'已配置' if os.environ.get('IWENCAI_API_KEY') else '未配置'}")
print(f"国信API Key: {'已配置' if os.environ.get('GS_API_KEY') else '未配置'}")

if not os.environ.get('MX_APIKEY') and not os.environ.get('IWENCAI_API_KEY'):
    print("\n⚠️ 警告: 至少需要配置一个数据源API Key")
    print("请在.env文件中配置: MX_APIKEY 或 IWENCAI_API_KEY")

## 2. 初始化选股器

In [ ]:
from daily_stock_selector import DailyStockSelector

# 创建选股器实例
selector = DailyStockSelector()
print("✅ 选股器初始化成功")

## 3. 获取主力资金流入TOP 20

In [ ]:
# 获取主力资金流入TOP 20
top_stocks = selector.get_top_main_inflow(top_n=20)

print(f"\n=== 主力资金流入TOP {len(top_stocks)} ===")
for i, stock in enumerate(top_stocks[:10], 1):
    print(f"{i}. {stock['name']}({stock['code']}) ")
    print(f"   涨幅: {stock.get('change_pct', 'N/A')}%")
    print(f"   主力流入: {stock.get('main_inflow', 'N/A')}亿")
    print(f"   DDX: {stock.get('ddx', 'N/A')}")
    print()

## 4. 筛选DDX刚转正的股票

DDX刚转正可能是启动信号，值得关注。

In [ ]:
# 筛选DDX刚转正的股票
breakout_stocks = selector.screen_breakout_stocks(top_stocks)

print(f"\n=== DDX刚转正股票 ({len(breakout_stocks)}只) ===")
for stock in breakout_stocks[:5]:
    print(f"\n{stock['name']}({stock['code']})")
    print(f"  昨日DDX: {stock.get('ddx_yesterday', 'N/A')}")
    print(f"  今日DDX: {stock.get('ddx_today', 'N/A')}")
    print(f"  主力流入: {stock.get('main_inflow', 'N/A')}亿")
    print(f"  涨幅: {stock.get('change_pct', 'N/A')}%")

## 5. 分级推荐

In [ ]:
# 分级推荐
recommendations = selector.grade_stocks(top_stocks)

print("\n=== 分级推荐结果 ===")
for grade, stocks in recommendations.items():
    if stocks:
        print(f"\n【{grade}级】")
        for stock in stocks[:3]:
            print(f"  - {stock['name']}({stock['code']})")
            print(f"    涨幅: {stock.get('change_pct', 'N/A')}%")
            print(f"    主力流入: {stock.get('main_inflow', 'N/A')}亿")
            print(f"    10日DDX: {stock.get('ddx_10d', 'N/A')}")

## 6. 推送到钉钉（可选）

In [ ]:
# 推送到钉钉
push = input("是否推送到钉钉？(y/n): ")

if push.lower() == 'y':
    selector.push_to_dingtalk(recommendations)
    print("✅ 已推送到钉钉")
else:
    print("跳过推送")

## 7. 分析单只股票

In [ ]:
# 分析单只股票
stock_code = input("输入股票代码（如600519）: ")

if stock_code:
    analysis = selector.analyze_single_stock(stock_code)
    
    print(f"\n=== {analysis['name']}({stock_code}) 分析 ===")
    print(f"当前价: {analysis.get('current_price', 'N/A')}元")
    print(f"涨幅: {analysis.get('change_pct', 'N/A')}%")
    print(f"主力流入: {analysis.get('main_inflow', 'N/A')}亿")
    print(f"今日DDX: {analysis.get('ddx_today', 'N/A')}")
    print(f"10日DDX: {analysis.get('ddx_10d', 'N/A')}")
    print(f"\n推荐等级: {analysis.get('grade', 'N/A')}")
    print(f"建议: {analysis.get('suggestion', 'N/A')}")

## 总结

本案例展示了完整的选股流程：
1. 获取主力资金流入数据
2. 筛选DDX刚转正的股票
3. 分级推荐（A/B/C/D级）
4. 推送到钉钉
5. 分析单只股票

### 数据源优先级
- 妙想API（首选，无限制）
- 问财API（备用，每日有限）

### 选股条件
- A级：涨幅<3% + 10日DDX>0 + 主力流入>5000万
- B级：涨幅<3% + 今日主力流入>0
- C级：涨幅3-5% + 主力流入
- D级：不符合条件